In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import geopandas as gpd
import geosnap as gsp
import numpy as np
import overturemaps
from tobler.dasymetric import floor_area_dot_density

la = gsp.io.get_acs(gsp.DataStore(), msa_fips="31080", years=2021)

traditional Dasymetric refinement leaves much to be desired. For example we often rely on areal imagery and remote sensing data to exclude uninhabited areas from areal interpolation processes. These techniques improve over traditional areal interpoaltion but give only coarse refinements. With new data sources constantly availble, scholars have sought ways to improve these techniques @Comber2019a, e.g. @Liu2019a, @Zoraghein_2016 and @Jia2016 use parcel data, and @Sridharan2013 combine LIDAR data on building heights with other remotely-sensed data. While useful improvements, these are neither scalable nor particularly conceptually valid; parcel data are very difficult to collect at scale (and vary widely in quality) whereas building *volumes* are technically not the correct quantity by which weights should be applied--people do not inhabit continuous three-dimensional space. 

Instead, it makes more sense to weight by building *floor area*, as humans are confined to two dimensional floor space--which may be raised to multiple floors. This small insight gives a performant, accurate interpolation method thats both conceptually valid and easily conducted nearly anywhere in the globe thanks to new openly available 3d building footprint data. The technique is useful both for more accurate interpolation, as well as creating stunningly detailed maps of multidimensional segregation. The approach is straightforward and relies only on (1) the quality of building classification into residential and (2) a floor-to-floor height assumption used to convert building volumes into floor area totals.

Thus, we collect census data and building footprint data for our area of interest (the Los Angeles metro region in this case), then limit our building footpring dasymetric layer to residentail buildings or those with unknown classifications. We then use the overall building height of each building to stack a set of footprints on top of one another. From the perspective of spatial analysis software, these overlapping polygons appear exactly as a set of floors sitting atop one another. Following, we spatially join these floors to the population of the census polygon they fall inside, which reassigns the population spread over euclidean area to the same population spread over *residential floor area* inside each polygon. A detailed transfer into another geometric unit then needs only the existing area-weighted mechanics commonly applied in the literature.

In [ ]:
la_bldgs = overturemaps.record_batch_reader(
    "building", list(la.total_bounds), stac=True
).read_all()
la_bldgs = gpd.GeoDataFrame.from_arrow(la_bldgs)
la_bldgs = la_bldgs[la_bldgs.intersects(la.union_all())]

la_bldgs = la_bldgs.set_crs(4326)
la_bldgs = la_bldgs.to_crs(la_bldgs.estimate_utm_crs())

In [ ]:
# keep residential or unknown; i.e. filter out anything *known* as non-residential

subtypes = ['residential', np.nan, None]

In [ ]:

#restypes = ["dormitory", "house", "apartments", "semidetached_house", None, np.nan]
#dc_bldgs = dc_bldgs[dc_bldgs["class"].isin(restypes)]

In [ ]:
la_bldgs = la_bldgs[la_bldgs["subtype"].isin(subtypes)]

In [ ]:
la_bldgs['height'] = la_bldgs['height'].fillna(3)

In [ ]:
from geosnap.visualize import lbexplore

In [ ]:
cols = ['n_nonhisp_white_persons', 'n_nonhisp_black_persons', 'n_hispanic_persons', 'n_asian_persons']

In [ ]:
la_pop_pts = floor_area_dot_density(
    la.set_index("geoid").to_crs(la_bldgs.crs),
    la_bldgs,
    columns=cols,
    height_col="height",
    floors_col='num_floors',
    floor_height=3,
    scale=0.2,
)

In [ ]:
la_pop_pts.sample(frac=0.5).to_crs(4326).lb.explore('category', categorical=True, cmap='rainbow').to_html("./la_race_density.html")

In [ ]:
la_pop_pts.groupby('category').count()

In [ ]:
la[cols].sum()

In [ ]:
from geosnap.visualize import lbexplore

In [ ]:
la_pop_pts.sample(frac=1).lb.explore(column='category', categorical=True, cmap='rainbow')